In [1]:
import os, glob, torch
!rm -rf /tmp/Siformer && git clone -q https://github.com/mpuu00001/Siformer /tmp/Siformer
os.chdir('/tmp/Siformer')
s = open('train.py').read()
s = s.replace('+ str(i) + ".pth")', '+ str(i) + ".pth", weights_only=False)')
s = s.replace('_, _, top_val_acc = evaluate_top_k(slr_model, val_loader, device)', 'pass')
open('train.py', 'w').write(s)
for d in ['datasets', 'normalization', 'augmentations', 'siformer']:
    p = f'/tmp/Siformer/{d}'
    if os.path.isdir(p) and not os.path.exists(p + '/__init__.py'):
        open(p + '/__init__.py', 'w').close()
!grep -n "weights_only" train.py
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

294:                    "out-checkpoints/" + args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i) + ".pth", weights_only=False)
GPU: True Tesla T4


In [2]:
hits = glob.glob('/kaggle/input/**/csv_rect_dip/train_smote.csv', recursive=True)
if not hits:   # nếu Kaggle không tự giải nén thì giải nén ở đây
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]
    !mkdir -p /kaggle/temp/data && unzip -q -o "{z}" -d /kaggle/temp/data
    hits = glob.glob('/kaggle/temp/data/**/csv_rect_dip/train_smote.csv', recursive=True)
ROOT = os.path.dirname(os.path.dirname(hits[0]))
print('ROOT =', ROOT)
for d in ('csv', 'csv_rect_dip', 'csv_rect_full'):
    print(d, sorted(os.listdir(f'{ROOT}/{d}')))

ROOT = /kaggle/input/datasets/uynlnhnh/dath-data
csv ['test.csv', 'train_smote.csv', 'val.csv']
csv_rect_dip ['test.csv', 'train_smote.csv', 'val.csv']
csv_rect_full ['test.csv', 'train_smote.csv', 'val.csv']


In [3]:
import re, shutil
EPOCHS = 100
RUNS = [('siformer', 'csv_rect_dip', 1), ('siformer', 'csv_rect_dip', 2),
        ('spoter', 'csv_rect_dip', 0), ('spoter', 'csv_rect_dip', 1), ('spoter', 'csv_rect_dip', 2),
        ('siformer', 'csv', 1), ('siformer', 'csv', 2)]
OUT = '/kaggle/working/results'

def pick_best(name):   # chọn checkpoint theo VAL, chép thành best.pth
    log = open(f'{name}.log', encoding='utf-8', errors='ignore').read()
    val = {int(e): float(a) for e, a in re.findall(r'\[(\d+)\]\s*VALIDATION\s+acc:\s*([\d.]+)', log)}
    saved = {int(e): f for e, f in re.findall(r'Save checkpoint for \[(\d+)\] as .*?(checkpoint_v_\d+\.pth)', log)}
    be = max(saved, key=lambda e: (val.get(e, 0), -e))
    shutil.copy(f'out-checkpoints/{name}/{saved[be]}', 'best.pth')
    print(f'>>> {name}: val cao nhất {val[be]:.4f} ở epoch {be} ({saved[be]}) -> best.pth')

for model, data, seed in RUNS:
    name = f"{model}_{data}_s{seed}"
    os.makedirs(f"{OUT}/{name}", exist_ok=True); os.chdir(f"{OUT}/{name}")
    CSV = f"{ROOT}/{data}"
    fim = "" if model == "spoter" else "True"
    !PYTHONPATH=/tmp/Siformer python -m train --experiment_name {name} --num_classes 100 --num_worker 4 --seed {seed} --epochs {EPOCHS} --FIM "{fim}" --training_set_path "{CSV}/train_smote.csv" --validation_set from-file --validation_set_path "{CSV}/val.csv" --testing_set_path "{CSV}/test.csv" 2>&1 | tee {name}.log
    pick_best(name)
os.chdir('/kaggle/working')
!zip -qr best_and_logs.zip results -i "*.log" "*best.pth"
!ls -la /kaggle/working

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
Cuda is available: True
Feature isolated transformer
self.selected_attn prob
Encoder with input adaptive
self.selected_attn prob
Encoder with input adaptive
self.selected_attn prob
Encoder with input adaptive
Using custom DecoderLayer
Normal decoder
num_enc_layers 3, num_dec_layers 2, patient 1
Starting siformer_csv_rect_dip_s1...
Training using /kaggle/input/datasets/uynlnhnh/dath-data/csv_rect_dip/train_smote.csv...
Validation using /kaggle/input/datasets/uynlnhnh/dath-data/csv_rect_dip/val.csv...


Save checkpoint for [1] as out-checkpoints/siformer_csv_rect_dip_s1/checkpoint_v_0.pth
[1] TRAIN  loss: 4.721376953125 acc: 0.011666666666666667
[1] AVG TRAIN time per sample (sec): 0.267706460